In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from datetime import datetime
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Setup project path
project_root = os.path.abspath(os.getcwd())
if project_root not in sys.path:
    sys.path.append(project_root)

print("Environment initialized")
print(f"Project root: {project_root}")
print(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

✓ Environment initialized
✓ Project root: /home/nao/synthetic-tabular-data
✓ Timestamp: 2026-07-18 16:06:59


In [ ]:
DEMO_CONFIG = {
    # Dataset selection
    "dataset": "telco_customer_churn",        # Options: telco_customer_churn | adult_income
    
    # Models to run (loop through all)
    "models_to_run": ["diffusion", "ctgan", "ctvae"],  # Run all 3 models sequentially
    
    # Privacy settings
    "enable_dp": False,
    "epsilon": 3.0,                           # Privacy budget: 1.5 (max) | 3.0 (balanced) | 10.0 (utility-focused)
    "delta": 1e-5,
    
    # Synthetic data generation
    "n_synthetic_rows": 1000,
    
    # Execution mode
    "execution_mode": "simulation",           # Options: simulation (3 sec) | training (5-15 min per model)
    
    # Demonstrations to run
    "run_model_comparison": False,            # Skip redundant comparison (already running 3 models)
    "run_privacy_sweep": False,               # Skip privacy sweep for time
}

# Data paths
if DEMO_CONFIG["dataset"] == "telco_customer_churn":
    data_path = os.path.join("data", "Telco-Customer-Churn.csv")
else:
    data_path = os.path.join("data", "adult", "adult.data")

print("\n" + "="*80)
print("DEMO CONFIGURATION")
print("="*80)
print(f"  {'dataset':.<40} {DEMO_CONFIG['dataset']}")
print(f"  {'models_to_run':.<40} {DEMO_CONFIG['models_to_run']}")
print(f"  {'execution_mode':.<40} {DEMO_CONFIG['execution_mode']}")
print(f"  {'data_path':.<40} {data_path}")
print("="*80)
print(f"\n⏳ Will run {len(DEMO_CONFIG['models_to_run'])} models sequentially with separate evaluation reports.\n")



DEMO CONFIGURATION
  dataset................................. telco_customer_churn
  models_to_run........................... ['diffusion', 'ctgan', 'ctvae']
  execution_mode.......................... simulation
  data_path............................... data/Telco-Customer-Churn.csv

⏳ Will run 3 models sequentially with separate evaluation reports.



In [ ]:
from src.inference.sampler import SyntheticSampler

print("\n" + "="*80)
print("INFERENCE DEMONSTRATION")
print("="*80)

# Example: Generate synthetic data from a single model
model_type = "diffusion"  # Options: diffusion, ctgan, ctvae
dataset = "telco_customer_churn"  # Options: telco_customer_churn, adult_income

print(f"\n[Step 1] Initialize SyntheticSampler")
sampler = SyntheticSampler(
    model_type=model_type,
    dataset_name=dataset,
    artifacts_root="artifacts"  # Thêm tham số artifacts_root
)
print(f"  ✓ Sampler initialized for model: {model_type}")

print(f"\n[Step 2] Load pre-trained model checkpoint")
# Xác định đường dẫn chính xác tới checkpoint và pipeline của mô hình
checkpoint_path = os.path.join("artifacts", dataset, f"{model_type}_nodp", "checkpoints", f"{model_type}_model.pt")
pipeline_path = os.path.join("artifacts", dataset, f"{model_type}_nodp", "preprocessing_pipeline.joblib")

sampler.load(checkpoint_path=checkpoint_path, pipeline_path=pipeline_path)
print(f"  ✓ Model loaded from checkpoint")

print(f"\n[Step 3] Generate synthetic data")
n_rows = 500  # Number of synthetic rows to generate
df_synthetic = sampler.generate(n_rows=n_rows)
print(f"  ✓ Generated {df_synthetic.shape[0]} synthetic rows × {df_synthetic.shape[1]} features")

# Display first few rows of synthetic data
print(f"\n[Result] First 5 rows of synthetic data:")
print(df_synthetic.head())

# Save to CSV (optional)
inference_csv_path = f"synthetic_data_{model_type}_{dataset}.csv"
df_synthetic.to_csv(inference_csv_path, index=False)
print(f"\n  ✓ Synthetic data saved to: {inference_csv_path}")

print("\n" + "="*80)
print("Synthetic data ready for use")
print("="*80)


INFERENCE DEMONSTRATION

[Step 1] Initialize SyntheticSampler


TypeError: SyntheticSampler.__init__() missing 1 required positional argument: 'artifacts_root'